# example / static / constant

The smallest complete model: one constant, to prove the register and pull path end to end.

| Term | In This Notebook |
|---|---|
| **Intent** | Prove that a project can be written, registered, pulled back and verified without any fitting |
| **Project Flavour** | `static` (`model.md` 3.2) |
| **Data Splits** | `golden` the one input and expected output, and a request that must give a signature error; every other split is unused |

## Setup

- **Stage:** none, preparation for the stages below.
- **Intent:** put `workbench` on the path, load `env`, and declare the model name, version and dataset names as constants (`model.md` 6.6).

In [ ]:
import json
import tempfile
from os.path import join

from workbench.data import golden_data
from workbench.env import init_env
from workbench.project import provenance, record_project, scratch_project_dir

env = init_env()

WORKSPACE = "example/static/constant"
MODEL_NAME = "example_static_constant"
TARGETS = [{"kind": "served"}, {"kind": "embedded", "language": "python"}, {"kind": "embedded", "language": "go"}, {"kind": "embedded", "language": "rust"}]
GOLDEN = "static_constant_golden"
CONSTANTS = {"example_static_out": 7}

## Package

- **Stage:** 5, package.
- **Intent:** write the `golden` rows, build the signature and `golden.json`, write a `static` project with one constant, verify it and replay the golden through the library loader (R1, R3).

In [ ]:
from workbench.model import build_signature, package_project, pull_model, register_preview, signature_input, signature_output, verify_project
from workbench.replay import require_replay

golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, lambda: {"example_static_in": [1.0, None], "expect": ["ok", "error"]})

signature = build_signature(
    MODEL_NAME,
    [signature_input("example_static_in", "none", "Unused input.")],
    [signature_output("example_static_out", "none", "A constant.")],
)
project_dir = scratch_project_dir(WORKSPACE)
manifest = package_project(
    project_dir,
    MODEL_NAME,
    "static",
    signature,
    golden_pl.iter_rows(named=True),
    provenance("manual", "none", [golden_path], "register path smoke test"),
    constants=CONSTANTS,
)
print(json.dumps(manifest, indent=2))

require_replay(project_dir)
metadata = record_project(WORKSPACE, project_dir, TARGETS)
print("replayed the golden sequences exactly through the shipped runtime")

## Register

- **Stage:** 6, register.
- **Intent:** register the verified project with the alias `preview`. A person sets `release` after review and the parity gate (R5).

In [ ]:
version = register_preview(project_dir, MODEL_NAME, env["MLFLOW_TRACKING_URI"], TARGETS, metadata["snapshot_version"])
print("registered version", version)

## Pull

- **Stage:** 8, pull (served path).
- **Intent:** pull `preview` back from the registry and check it equals the registered project, proving the round trip (R7).

In [ ]:
pulled_dir = join(tempfile.mkdtemp(), "pulled")
pulled = pull_model(MODEL_NAME, "preview", pulled_dir, env["MLFLOW_TRACKING_URI"])
assert pulled == verify_project(project_dir), "the pulled project differs from the registered one"
print("pulled", pulled["name"], "matches the registered project")